# MRMS Data Processing - Filtered free cells

In [1]:
import xarray as xr
import pandas as pd

# Define the months and year to process
year = "2024"
months = ["05", "06", "07", "08", "09"]

for month in months:
    print(f"\n=== Processing for {year}{month} ===")
    
    # Construct file names based on month
    netcdf_file = f"merge_split_{year}{month}_1km.nc"  # NetCDF file for the month
    initiation_file = f"all_initiation_points_{year}{month}_1km.csv"
    tracks_file = f"all_tracks_{year}{month}_1km.csv"
    
    cell_status_file = f"cell_variables_status_{year}{month}_1km.csv"
    merged_output_file = f"merged_track_init_{year}{month}_1km.csv"
    final_output_file = f"final_merged_track_init_with_cell_status_{year}{month}_1km.csv"
    filtered_output_file = f"filtered_free_cells_{year}{month}_1km.csv"
    
    # -------------------------------
    # Step 1: Process NetCDF file to create cell status CSV
    # -------------------------------
    try:
        # Open the dataset
        ds = xr.open_dataset(netcdf_file)
        
        # Select only the cell-based variables
        ds_subset = ds[
            [
                "cell_parent_track_id",
                "cell_child_feature_count",
                "cell_starts_with_split",
                "cell_ends_with_merge",
            ]
        ]
        
        # Convert to a Pandas DataFrame and reset the index
        df = ds_subset.to_dataframe().reset_index()
        
        # Create status columns
        df["Merge"] = df["cell_ends_with_merge"].apply(lambda x: "Merge" if x else "")
        df["Split"] = df["cell_starts_with_split"].apply(lambda x: "Split" if x else "")
        df["Both"] = df.apply(
            lambda row: "Both" if row["cell_starts_with_split"] and row["cell_ends_with_merge"] else "",
            axis=1
        )
        df["Free"] = df.apply(
            lambda row: "Free" if not row["cell_starts_with_split"] and not row["cell_ends_with_merge"] else "",
            axis=1
        )
        
        # Choose columns for the output CSV
        output_columns = [
            "cell",
            "cell_parent_track_id",
            "cell_child_feature_count",
            "cell_starts_with_split",
            "cell_ends_with_merge",
            "Merge",
            "Split",
            "Both",
            "Free",
        ]
        
        # Save cell status data to CSV
        df[output_columns].to_csv(cell_status_file, index=False)
        print(f"Step 1: Cell-based variables and status saved to '{cell_status_file}'")
    except Exception as e:
        print(f"Error in Step 1 (NetCDF processing) for {year}{month}: {e}")
        continue  # Skip to next month if there's an error
    
    
    # -------------------------------
    # Step 2: Merge initiation points, tracks, and cell status
    # -------------------------------
    try:
        # Load the initiation and tracks CSV files
        df_initiation = pd.read_csv(initiation_file)
        df_tracks = pd.read_csv(tracks_file)
        
        # Convert 'time' column to datetime and extract date only
        df_initiation['time'] = pd.to_datetime(df_initiation['time'])
        df_tracks['time'] = pd.to_datetime(df_tracks['time'])
        df_initiation['date'] = df_initiation['time'].dt.date
        df_tracks['date'] = df_tracks['time'].dt.date
        
        # Merge the two DataFrames on 'feature' and 'date'
        merged_df = pd.merge(df_initiation, df_tracks, on=['feature', 'date'], how="inner")
        
        # Remove duplicate columns if they exist (those ending with _x or _y)
        for col in list(merged_df.columns):
            if col.endswith("_x") or col.endswith("_y"):
                base_col = col[:-2]
                if base_col in merged_df.columns:
                    merged_df.drop(columns=[col], inplace=True)
                else:
                    merged_df.rename(columns={col: base_col}, inplace=True)
        
        # Save the merged initiation and tracks CSV
        merged_df.to_csv(merged_output_file, index=False)
        print(f"Step 2: Merged initiation and tracks saved as '{merged_output_file}' with {len(merged_df)} rows.")
        
        # Load the cell status dataset
        df_cell_status = pd.read_csv(cell_status_file)
        
        # Rename "None" column to "Free" if it exists (should already be "Free")
        if "None" in df_cell_status.columns:
            df_cell_status.rename(columns={"None": "Free"}, inplace=True)
        # Replace empty values in "Free" column with "Free"
        if "Free" in df_cell_status.columns:
            df_cell_status["Free"] = df_cell_status["Free"].replace("", "Free")
        
        # Merge cell status information with the merged DataFrame on 'cell'
        final_df = pd.merge(merged_df, df_cell_status, on="cell", how="inner")
        
        # Save the final merged DataFrame
        final_df.to_csv(final_output_file, index=False)
        print(f"Step 2: Final merged CSV saved as '{final_output_file}' with {len(final_df)} rows.")
    except FileNotFoundError as e:
        print(f"Error in Step 2 (File not found) for {year}{month}: {e}")
        continue
    except pd.errors.EmptyDataError:
        print(f"Error in Step 2: One of the CSV files is empty for {year}{month}.")
        continue
    except Exception as e:
        print(f"Unexpected error in Step 2 for {year}{month}: {e}")
        continue
    
    
    # -------------------------------
    # Step 3: Filter rows where the 'Free' column is 'Free'
    # -------------------------------
    try:
        # Load the final merged CSV file
        df_final = pd.read_csv(final_output_file)
        
        # Filter rows where the 'Free' column equals "Free"
        filtered_df = df_final[df_final['Free'] == 'Free']
        
        # Save the filtered data to a new CSV file
        filtered_df.to_csv(filtered_output_file, index=False)
        print(f"Step 3: Filtered free cells saved as '{filtered_output_file}' with {len(filtered_df)} rows.")
    except Exception as e:
        print(f"Error in Step 3 (Filtering) for {year}{month}: {e}")


C:\Users\msmillan\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



=== Processing for 202405 ===


C:\Users\msmillan\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


Step 1: Cell-based variables and status saved to 'cell_variables_status_202405_1km.csv'
Step 2: Merged initiation and tracks saved as 'merged_track_init_202405_1km.csv' with 70023 rows.
Step 2: Final merged CSV saved as 'final_merged_track_init_with_cell_status_202405_1km.csv' with 3671 rows.
Step 3: Filtered free cells saved as 'filtered_free_cells_202405_1km.csv' with 2458 rows.

=== Processing for 202406 ===
Step 1: Cell-based variables and status saved to 'cell_variables_status_202406_1km.csv'
Step 2: Merged initiation and tracks saved as 'merged_track_init_202406_1km.csv' with 127151 rows.
Step 2: Final merged CSV saved as 'final_merged_track_init_with_cell_status_202406_1km.csv' with 28001 rows.
Step 3: Filtered free cells saved as 'filtered_free_cells_202406_1km.csv' with 11705 rows.

=== Processing for 202407 ===
Step 1: Cell-based variables and status saved to 'cell_variables_status_202407_1km.csv'
Step 2: Merged initiation and tracks saved as 'merged_track_init_202407_1km.csv

In [1]:
# -*- coding: utf-8 -*-
import pandas as pd

# 2024 months only
year = "2024"
months = ["05", "06", "07", "08", "09"]

total_cells_all = 0
free_cells_all = 0

results = []

for month in months:
    cell_status_file = f"cell_variables_status_{year}{month}_1km.csv"
    
    try:
        df = pd.read_csv(cell_status_file)

        total_cells = df["cell"].nunique()
        free_cells = df[df["Free"] == "Free"]["cell"].nunique()
        percent_free = (free_cells / total_cells) * 100 if total_cells > 0 else 0

        results.append({
            "month": f"{year}{month}",
            "total_cells": total_cells,
            "free_cells": free_cells,
            "percent_free": percent_free
        })

        total_cells_all += total_cells
        free_cells_all += free_cells

        print(f"{year}{month}: {percent_free:.2f}% "
              f"(Free cells = {free_cells}, Total cells = {total_cells})")

    except Exception as e:
        print(f"Error processing {cell_status_file}: {e}")

# Overall percentage for all 2024 months combined
overall_percent = (free_cells_all / total_cells_all) * 100 if total_cells_all > 0 else 0

print("\n=== Overall for 2024 May-Sep ===")
print(f"Overall Free Cell Percentage: {overall_percent:.2f}%")
print(f"Total Free Cells = {free_cells_all}")
print(f"Total Cells = {total_cells_all}")

# Save summary
df_results = pd.DataFrame(results)

overall_row = pd.DataFrame([{
    "month": "2024_MaySep_Total",
    "total_cells": total_cells_all,
    "free_cells": free_cells_all,
    "percent_free": overall_percent
}])

df_results = pd.concat([df_results, overall_row], ignore_index=True)
df_results.to_csv("free_cell_percentage_2024_may_sep.csv", index=False)

C:\Users\msmillan\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


202405: 66.52% (Free cells = 731, Total cells = 1099)
202406: 42.05% (Free cells = 3112, Total cells = 7401)
202407: 45.87% (Free cells = 4806, Total cells = 10478)
202408: 48.41% (Free cells = 2915, Total cells = 6022)
202409: 40.52% (Free cells = 3427, Total cells = 8458)

=== Overall for 2024 May-Sep ===
Overall Free Cell Percentage: 44.81%
Total Free Cells = 14991
Total Cells = 33458
